# Replit ↔ Colab Bridge — Backend Server

**Запустите все ячейки по порядку (Runtime → Run all).**

После запуска последней ячейки скопируйте URL и вставьте его в поле **Colab Connection** в интерфейсе Replit.

Туннель: **Cloudflare Tunnel** — бесплатно, без регистрации.

---

In [ ]:
# ============================================================
# Ячейка 1: Установка зависимостей
# ============================================================
!pip install -q flask flask-cors openai requests

In [ ]:
# ============================================================
# Ячейка 2: Скачиваем cloudflared (туннель без регистрации)
# ============================================================
import os

if not os.path.exists('./cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
    print('✓ cloudflared скачан')
else:
    print('✓ cloudflared уже есть')

In [ ]:
# ============================================================
# Ячейка 3: Введите ваш OpenRouter API ключ
# ============================================================
import getpass

OPENROUTER_API_KEY = getpass.getpass('Введите ваш OpenRouter API ключ (openrouter.ai/keys): ')
print('✓ Ключ получен')

In [ ]:
# ============================================================
# Ячейка 4: Flask-сервер
# ============================================================
import threading
import time
import requests as req_lib
from flask import Flask, request, jsonify
from flask_cors import CORS
from openai import OpenAI

app = Flask(__name__)
CORS(app)

openrouter_client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'version': '2.0'})


@app.route('/models/free', methods=['GET'])
def get_free_models():
    try:
        resp = req_lib.get(
            'https://openrouter.ai/api/v1/models',
            headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}'},
            timeout=10
        )
        resp.raise_for_status()
        all_models = resp.json().get('data', [])
        free_models = []
        for m in all_models:
            pricing = m.get('pricing', {})
            prompt_price = str(pricing.get('prompt', '1'))
            completion_price = str(pricing.get('completion', '1'))
            is_free = (
                prompt_price in ('0', '0.0', '') and
                completion_price in ('0', '0.0', '')
            ) or m.get('id', '').endswith(':free')
            if is_free:
                free_models.append({
                    'id': m.get('id', ''),
                    'name': m.get('name', m.get('id', '')),
                    'description': m.get('description', ''),
                    'context_length': m.get('context_length', 0),
                    'pricing': pricing,
                })
        return jsonify({'status': 'ok', 'models': free_models, 'count': len(free_models)})
    except Exception as e:
        return jsonify({'status': 'error', 'message': str(e), 'models': [], 'count': 0}), 500


def fetch_model_card(model_id):
    try:
        resp = req_lib.get(
            f'https://openrouter.ai/api/v1/models/{model_id}',
            headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}'},
            timeout=5
        )
        if resp.status_code == 200:
            m = resp.json()
            return {
                'id': m.get('id', model_id),
                'name': m.get('name', model_id),
                'description': m.get('description', ''),
                'context_length': m.get('context_length'),
                'pricing': m.get('pricing', {}),
            }
    except Exception:
        pass
    return {'id': model_id, 'name': model_id}


def run_single(model_id, messages, temperature, max_tokens):
    kwargs = {
        'model': model_id or 'openrouter/auto',
        'messages': messages,
        'temperature': temperature,
        'max_tokens': max_tokens,
    }
    t_start = time.time()
    response = openrouter_client.chat.completions.create(**kwargs)
    elapsed = round(time.time() - t_start, 3)
    result_text = response.choices[0].message.content or ''
    model_used = response.model or model_id or 'unknown'
    usage = response.usage
    return {
        'result': result_text,
        'model': model_used,
        'metrics': {
            'prompt_tokens': usage.prompt_tokens if usage else 0,
            'completion_tokens': usage.completion_tokens if usage else 0,
            'total_tokens': usage.total_tokens if usage else 0,
            'execution_time': elapsed,
        }
    }


@app.route('/run', methods=['POST'])
def run_experiment():
    data = request.get_json()
    if not data:
        return jsonify({'status': 'error', 'message': 'No JSON body'}), 400

    task_type = data.get('task_type', 'text_generation')
    mode = data.get('mode', 'auto')
    model_id = data.get('model')
    models_list = data.get('models', [])
    params = data.get('params', {})

    prompt = params.get('prompt', '')
    temperature = float(params.get('temperature', 0.7))
    max_tokens = int(params.get('max_tokens', 1024))
    system_prompt = params.get('system_prompt')
    custom_code = params.get('custom_code')

    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})

    try:
        if task_type == 'model_comparison' and models_list:
            comparisons = []
            for mid in models_list:
                try:
                    r = run_single(mid, messages, temperature, max_tokens)
                    comparisons.append({
                        'model_id': mid, 'result': r['result'],
                        'prompt_tokens': r['metrics']['prompt_tokens'],
                        'completion_tokens': r['metrics']['completion_tokens'],
                        'total_tokens': r['metrics']['total_tokens'],
                        'execution_time': r['metrics']['execution_time'],
                        'error': None,
                    })
                except Exception as e:
                    comparisons.append({'model_id': mid, 'result': '', 'error': str(e)})
            return jsonify({'status': 'ok', 'task_type': task_type, 'mode': mode, 'comparisons': comparisons})

        if task_type == 'custom_code' and custom_code:
            import io, contextlib
            namespace = {'openrouter_client': openrouter_client, 'OPENROUTER_API_KEY': OPENROUTER_API_KEY}
            stdout_capture = io.StringIO()
            t_start = time.time()
            with contextlib.redirect_stdout(stdout_capture):
                exec(custom_code, namespace)
            elapsed = round(time.time() - t_start, 3)
            return jsonify({
                'status': 'ok', 'task_type': task_type, 'mode': mode,
                'result': stdout_capture.getvalue(),
                'metrics': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'execution_time': elapsed},
            })

        r = run_single(model_id, messages, temperature, max_tokens)
        model_used = r['model']
        model_card = fetch_model_card(model_used)
        return jsonify({
            'status': 'ok', 'task_type': task_type, 'mode': mode,
            'result': r['result'], 'model': model_used,
            'model_card': model_card, 'metrics': r['metrics'],
        })

    except Exception as e:
        return jsonify({'status': 'error', 'task_type': task_type, 'mode': mode, 'message': str(e)}), 500


flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False, debug=False)
)
flask_thread.daemon = True
flask_thread.start()
time.sleep(2)
print('✓ Flask-сервер запущен на порту 5000')

In [ ]:
# ============================================================
# Ячейка 5: Запуск Cloudflare Tunnel (без регистрации)
# ============================================================
import subprocess
import threading
import re
import time

tunnel_url = None

def start_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break

t = threading.Thread(target=start_tunnel)
t.daemon = True
t.start()

# Ждём пока туннель откроется (до 30 секунд)
for _ in range(30):
    if tunnel_url:
        break
    time.sleep(1)

if tunnel_url:
    print('\n' + '='*60)
    print('  СЕРВЕР ГОТОВ')
    print('='*60)
    print(f'\n  URL для Replit:')
    print(f'\n  >>> {tunnel_url} <<<')
    print(f'\n  Скопируйте этот URL в поле Colab Connection')
    print(f'  в интерфейсе Replit (Notebook UI или Bridge).')
    print('\n' + '='*60)
    print('  Не закрывайте эту вкладку!')
    print('='*60)
else:
    print('Туннель не открылся за 30 секунд. Попробуйте перезапустить ячейку.')

## Доступные эндпоинты

| Метод | Путь | Описание |
|-------|------|----------|
| GET | `/health` | Проверка статуса |
| GET | `/models/free` | Список бесплатных моделей OpenRouter |
| POST | `/run` | Запуск эксперимента |

## Важно

- Сессия Colab живёт ~12 часов (бесплатный план)
- Cloudflare Tunnel — бесплатно, без аккаунта, работает до закрытия сессии
- При истечении сессии — перезапустите ячейки и вставьте новый URL